<a href="https://colab.research.google.com/github/Baonguyen2414l/cmu-movie-recommender/blob/Baodev/notebooks/01_eda_and_preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 1. Cài đặt  thư viện chia dữ liệu đa nhãn (Multi-label Stratification)
!pip install -q iterative-stratification
# Chạy ở một cell riêng để cài thư viện chia tập dữ liệu đa nhãn
!pip install iterative-stratification nltk -q
# 2. Thư viện hệ thống & chuẩn Python (Standard Libraries)
import os
import re
import json
import pickle
from collections import Counter

# 3. Thư viện xử lý dữ liệu & vẽ biểu đồ (Third-party Libraries)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 4. Thư viện NLP, Machine Learning & Google Colab
import nltk
from nltk.corpus import stopwords
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from google.colab import drive
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [8]:


# 1. Kết nối Google Drive
drive.mount('/content/drive')
# Cập nhật đúng đường dẫn thư mục lưu file của bạn
data_path = '/content/drive/MyDrive/MovieRec_Shared_Data/Dataset/'

# Đọc file danh mục phim
df = pd.read_csv(data_path + 'movies_catalog_clean.csv')
print(f"Đã đọc thành công {len(df)} phim!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã đọc thành công 2353 phim!


# **Tách năm phát hành và Làm sạch văn bản (raw_text & clean_text)**

In [9]:


# Tải tài nguyên xử lý ngôn ngữ tự nhiên từ NLTK
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# 1. Đọc file danh mục phim đã lọc (2353 phim)
df = pd.read_csv(data_path + 'movies_catalog_clean.csv')

# 2. Tách Năm phát hành (release_year) và Tên phim sạch (title_clean) từ cột title_raw
# Ví dụ: "Toy Story (1995)" -> title_clean: "Toy Story", release_year: 1995
def extract_title_and_year(title_raw):
    match = re.search(r'^(.*)\s+\((\d{4})\)\s*$', str(title_raw))
    if match:
        return match.group(1).strip(), int(match.group(2))
    return str(title_raw).strip(), np.nan

df[['title_clean', 'release_year']] = df['title_raw'].apply(
    lambda x: pd.Series(extract_title_and_year(x))
)

# 3. Tạo cột raw_text (Dành cho Bi-LSTM và LLM / Sentence Transformer)
# Giữ nguyên cấu trúc câu, dấu câu và chữ hoa/thường để bảo toàn ngữ cảnh, chỉ xóa khoảng trắng thừa
def make_raw_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 4. Tạo cột clean_text (Dành cho TF-IDF và Doc2Vec)
# Chuyển chữ thường, bỏ ký tự đặc biệt/số, lọc từ dừng (stopwords) và đưa từ về gốc (Lemmatization)
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def make_clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text) # Chỉ giữ lại chữ cái tiếng Anh
    tokens = text.split()
    clean_tokens = [
        lemmatizer.lemmatize(w) for w in tokens
        if w not in stop_words and len(w) > 2
    ]
    return ' '.join(clean_tokens)

print("Đang xử lý văn bản...")
df['raw_text'] = df['overview'].apply(make_raw_text)
df['clean_text'] = df['overview'].apply(make_clean_text)

print("Hoàn tất làm sạch! Kiểm tra 2 dòng đầu:")
display(df[['movie_id', 'title_clean', 'release_year', 'raw_text', 'clean_text']].head(2))

Đang xử lý văn bản...
Hoàn tất làm sạch! Kiểm tra 2 dòng đầu:


,movie_id,title_clean,release_year,raw_text,clean_text
0,1,Toy Story,1995,"Led by Woody, Andy's toys live happily in his ...",led woody andy toy live happily room andy birt...
1,2,Jumanji,1995,When siblings Judy and Peter discover an encha...,sibling judy peter discover enchanted board ga...


Chia tập dữ liệu 70% Train – 15% Val – 15% Test (Multilabel Stratification)**văn bản in đậm**

In [10]:


# 1. Biến đổi cột genres (dạng 'Animation|Children's|Comedy') thành ma trận nhị phân đa nhãn (One-hot)
df['genre_list'] = df['genres'].apply(lambda x: str(x).split('|'))
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])

print(f"Tổng số thể loại chuẩn hóa: {len(mlb.classes_)} -> {list(mlb.classes_)}")

# 2. Bước chia 1: Tách 70% Train và 30% Temp (Val + Test)
msss_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
for train_idx, temp_idx in msss_1.split(df, genre_matrix):
    pass

# 3. Bước chia 2: Chia đôi tập Temp (30%) thành 15% Val và 15% Test (tỷ lệ 50/50 của Temp)
temp_genre_matrix = genre_matrix[temp_idx]
msss_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
for val_rel_idx, test_rel_idx in msss_2.split(df.iloc[temp_idx], temp_genre_matrix):
    val_idx = temp_idx[val_rel_idx]
    test_idx = temp_idx[test_rel_idx]

# 4. Gán nhãn phân tập vào cột 'split' và reset lại index chuẩn từ 0 -> 2352
df['split'] = 'train'
df.loc[val_idx, 'split'] = 'val'
df.loc[test_idx, 'split'] = 'test'

# Xóa cột tạm genre_list trước khi lưu
df.drop(columns=['genre_list'], inplace=True)
df = df.reset_index(drop=True)

print("\nThống kê số lượng phim sau khi chia:")
print(df['split'].value_counts())
print("\nTỷ lệ phần trăm:")
print((df['split'].value_counts(normalize=True) * 100).round(2))

Tổng số thể loại chuẩn hóa: 18 -> ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

Thống kê số lượng phim sau khi chia:
split
train    1649
val       361
test      343
Name: count, dtype: int64

Tỷ lệ phần trăm:
split
train    70.08
val      15.34
test     14.58
Name: proportion, dtype: float64


In [12]:
# 1. Kiểm tra phân phối thể loại giữa Train / Val / Test (Dùng làm bảng số liệu cho Báo cáo)
genre_dist = pd.DataFrame(index=mlb.classes_)
for split_name in ['train', 'val', 'test']:
    sub_df = df[df['split'] == split_name]
    counts = sub_df['genres'].str.get_dummies(sep='|').sum()
    genre_dist[f'{split_name}_count'] = counts
    genre_dist[f'{split_name}_%'] = (counts / len(sub_df) * 100).round(1)

print("Bảng đối chiếu phân phối thể loại trên 3 tập (Multilabel Stratification):")
display(genre_dist)

# 2. Lưu file Master và 3 file thành phần vào Google Drive
df.to_csv(data_path + 'movies_master.csv', index=False)
df[df['split'] == 'train'].to_csv(data_path + 'train.csv', index=False)
df[df['split'] == 'val'].to_csv(data_path + 'val.csv', index=False)
df[df['split'] == 'test'].to_csv(data_path + 'test.csv', index=False)

print("\n Đã lưu thành công bộ dữ liệu chuẩn hóa vào Google Drive:")
print(f" - {data_path}movies_master.csv (Dùng chung cho toàn nhóm - {len(df)} phim)")
print(f" - {data_path}train.csv ({len(df[df['split']=='train'])} phim)")
print(f" - {data_path}val.csv ({len(df[df['split']=='val'])} phim)")
print(f" - {data_path}test.csv ({len(df[df['split']=='test'])} phim)")

Bảng đối chiếu phân phối thể loại trên 3 tập (Multilabel Stratification):


,train_count,train_%,val_count,val_%,test_count,test_%
Action,230,13.9,49,13.6,50,14.6
Adventure,128,7.8,28,7.8,27,7.9
Animation,39,2.4,8,2.2,9,2.6
Children's,104,6.3,22,6.1,22,6.4
Comedy,549,33.3,118,32.7,118,34.4
Crime,100,6.1,22,6.1,21,6.1
Documentary,47,2.9,10,2.8,10,2.9
Drama,643,39.0,137,38.0,138,40.2
Fantasy,30,1.8,6,1.7,7,2.0
Film-Noir,20,1.2,5,1.4,4,1.2



 Đã lưu thành công bộ dữ liệu chuẩn hóa vào Google Drive:
 - /content/drive/MyDrive/MovieRec_Shared_Data/Dataset/movies_master.csv (Dùng chung cho toàn nhóm - 2353 phim)
 - /content/drive/MyDrive/MovieRec_Shared_Data/Dataset/train.csv (1649 phim)
 - /content/drive/MyDrive/MovieRec_Shared_Data/Dataset/val.csv (361 phim)
 - /content/drive/MyDrive/MovieRec_Shared_Data/Dataset/test.csv (343 phim)
